# Day 29 · Drift: notice when the emails change

**First-principles question:** How would you know, a month after go-live, that the agent is getting worse?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import numpy as np
import pandas as pd
from lab.acme import INTENTS, load_cases
from lab.agent import run_agent
from lab.evals import score

cases = load_cases()

## 1 · A shadow week
Write 60 new emails with a different mix. Start from your test cases and change the mix on purpose.

In [ ]:
rng = np.random.default_rng(1)
MIX = {"outage": 0.35, "billing": 0.2, "speed": 0.15, "cancel": 0.1, "upgrade": 0.1, "relocation": 0.05, "other": 0.05}
# TODO: build SHADOW: 60 cases in the format of cases.jsonl, drawn from `cases` by MIX, with at least 10 rewritten
#       in Hinglish and at least 5 mentioning a new plan name ("Ultra 2 Gbps").
SHADOW = []

## 2 · Replay and compare

In [ ]:
def mix(intents):
    s = pd.Series(intents).value_counts(normalize=True)
    return np.array([max(s.get(i, 0.0), 1e-4) for i in INTENTS])

def psi(expected, actual):
    """Population stability index between two distributions over the same buckets."""
    # TODO: sum((actual - expected) * ln(actual / expected))
    ...

shadow_out = [run_agent(c, run=200 + i)[0] for i, c in enumerate(SHADOW)]
test_mix, live_mix = mix([c["expected_intent"] for c in cases]), mix([o.get("intent") for o in shadow_out])
test_pass = np.mean([score(c, run_agent(c)[0])["passed"] for c in cases])
live_pass = np.mean([score(c, o)["passed"] for c, o in zip(SHADOW, shadow_out)])
print(f"PSI {psi(test_mix, live_mix):.3f} · pass rate test {test_pass:.0%} → shadow {live_pass:.0%}")

## 3 · Alert rules

In [ ]:
# TODO: each rule: (name, a function of the numbers that returns True when it should fire, who is told)
ALERTS = [
    ("intent mix changed", lambda: psi(test_mix, live_mix) > 0.25, "support lead"),
]
fired = [(name, who) for name, rule, who in ALERTS if rule()]
fired

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(SHADOW) >= 60, "Step 1: 60 shadow emails"
assert abs(psi(np.array([0.5, 0.5]), np.array([0.5, 0.5]))) < 1e-12, "Same distribution, PSI 0"
assert psi(np.array([0.9, 0.1]), np.array([0.1, 0.9])) > 0.25
assert len(ALERTS) >= 3 and fired, "Step 3: at least 3 rules, and one fires on the shadow week"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")